## 1. Importación de pandas y carga del conjunto de datos

En esta primera parte se importa la librería `pandas`, que permite leer y manipular datos en forma de tabla. Después se carga el archivo `ingresos.csv` dentro del DataFrame llamado `personas`.

Luego se imprimen varias muestras aleatorias del conjunto de datos usando `sample(frac=2/3, replace=True)`. Esto toma aproximadamente dos terceras partes de los registros y permite repetición de filas, simulando el principio de muestreo con reemplazo que se usa en los bosques aleatorios.


In [ ]:
import pandas as pd

personas = pd.read_csv("ingresos.csv")

print (personas.sample(frac=2/3, replace=True))
print (personas.sample(frac=2/3, replace=True))
print (personas.sample(frac=2/3, replace=True))
print (personas.sample(frac=2/3, replace=True))
print (personas.sample(frac=2/3, replace=True))

## 2. Selección aleatoria de variables predictoras

Aquí se importa la función `sample` desde el módulo `random`. Después se muestran las columnas predictoras del DataFrame, es decir, todas las columnas excepto la última.

La instrucción `personas.columns[:-1]` selecciona todas las columnas menos la variable objetivo `ingreso`. Posteriormente, `sample(list(personas.columns[:-1]), 3)` elige aleatoriamente tres columnas predictoras.

Esta parte ayuda a entender la lógica del bosque aleatorio: no siempre se usan todas las variables en cada división, sino subconjuntos aleatorios de características.


In [ ]:
from random import sample

print(personas.columns[:-1], "\n")
print(sample(list(personas.columns[:-1]), 3))

## 3. Verificación de la versión de scikit-learn

En este bloque se consulta la versión instalada de `scikit-learn`. Esto es importante porque algunos parámetros pueden cambiar entre versiones. Por ejemplo, en versiones actuales el parámetro correcto para limitar la proporción de muestras usadas por árbol es `max_samples`, no `max_sample`.


In [ ]:
import sklearn
print(sklearn.__version__)

## 4. Creación y entrenamiento del modelo RandomForestClassifier

En esta celda se importa `RandomForestClassifier`, que es el clasificador de Bosque Aleatorio de `scikit-learn`.

El modelo se configura con los siguientes parámetros principales:

- `n_estimators=100`: construye 100 árboles de decisión.
- `criterion="gini"`: usa el índice Gini para medir la calidad de las divisiones.
- `max_features="sqrt"`: en cada división considera aproximadamente la raíz cuadrada del número total de variables.
- `bootstrap=True`: activa el muestreo con reemplazo.
- `max_samples=2/3`: cada árbol se entrena con dos terceras partes de los datos.
- `oob_score=True`: calcula una evaluación interna usando los datos que quedaron fuera de cada muestra bootstrap.

Después, `bosque.fit(...)` entrena el modelo usando como variables predictoras todas las columnas excepto `ingreso`, y como variable objetivo la columna `ingreso`.

Finalmente, se imprimen tres resultados: una predicción individual, la precisión del modelo sobre los datos de entrenamiento y el puntaje OOB.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

bosque = RandomForestClassifier(n_estimators=100,
                                criterion="gini",
                                max_features="sqrt",
                                bootstrap=True,
                                max_samples=2/3,
                                oob_score=True)

bosque.fit(personas[personas.columns[:-1]].values, personas["ingreso"].values)

print (bosque.predict([[50, 16, 1, 1, 40]]))
print (bosque.score(personas[personas.columns[:-1]].values, personas["ingreso"].values))
print (bosque.oob_score_)

## 5. Visualización de los árboles del bosque

En esta última parte se importa `matplotlib.pyplot` para graficar y el módulo `tree` de `sklearn` para visualizar cada árbol de decisión.

El ciclo `for arbol in bosque.estimators_:` recorre todos los árboles entrenados dentro del bosque. Para cada árbol, `tree.plot_tree(...)` genera su representación gráfica y `plt.show()` la muestra en pantalla.

Esta visualización permite observar que el bosque no está compuesto por un único árbol, sino por muchos árboles con estructuras potencialmente diferentes.


In [ ]:
import matplotlib.pyplot as plt
from sklearn import tree

for arbol in bosque.estimators_:
  tree.plot_tree(arbol, feature_names=personas.columns[:-1])
  plt.show()

# Documentación final del tema y del código

El ejercicio muestra de manera práctica cómo funciona un Bosque Aleatorio aplicado a un problema de clasificación. Primero se carga un conjunto de datos, luego se explora la idea de tomar muestras aleatorias y seleccionar variables al azar. Posteriormente se construye un modelo con `RandomForestClassifier`, se entrena con las variables predictoras y se evalúa su desempeño.

El parámetro `bootstrap=True` permite que cada árbol trabaje con una muestra distinta del conjunto de datos. Esto es importante porque reduce la dependencia del modelo respecto de una sola partición de los datos. A su vez, `max_features="sqrt"` hace que cada árbol considere diferentes subconjuntos de variables, aumentando la diversidad entre árboles.

El uso de `oob_score=True` permite obtener una estimación adicional del rendimiento del modelo sin separar manualmente un conjunto de prueba. El valor `oob_score_` se calcula con las observaciones que no fueron incluidas en la muestra bootstrap de cada árbol.

En conclusión, el Bosque Aleatorio mejora la estabilidad de los árboles de decisión porque combina muchos modelos simples. En lugar de depender de una sola estructura de decisión, obtiene una predicción colectiva. Por eso suele ser útil en problemas de clasificación y regresión cuando se busca un modelo más robusto que un árbol individual.
